# Introduction to Probability — FT Draft v6

**Week 5 · Tuesday · Target: ~150 minutes + 30-minute buffer**

## Learning goals

By the end of this lesson, you should be able to:

1. Describe sample spaces, events, and conditional probability.
2. Distinguish theoretical probabilities from estimates based on observations.
3. Identify discrete and continuous random variables.
4. Choose between `pmf`, `pdf`, `cdf`, `sf`, and `ppf`.
5. Recognize when binomial, geometric, Poisson, exponential, and normal models are appropriate.
6. Translate a real-world question into a probability calculation using SciPy.

**How to work:** Follow each example, then complete the short **Check for Understanding** prompts. The final challenge combines several ideas.

## 0. Setup

We use **SciPy** to define probability distributions, generate random observations, and calculate probabilities. We use **NumPy** for numerical operations on the simulated results and **Matplotlib** for plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom, geom, poisson, expon, norm

In [ ]:
# Consistent visualization: blue = event, gray = complement.
# Solid directed lines mark included outcomes; dashed gaps mark excluded ones.
# SciPy supplies probabilities/densities; NumPy supplies plotting coordinates.
def draw_probability_span(ax, coordinates, mask, y, label, color, step=None):
    """Draw left-to-right accumulation, interrupting the line over excluded values."""
    coordinates = np.asarray(coordinates)
    mask = np.asarray(mask, dtype=bool)
    selected = np.flatnonzero(mask)
    if not len(selected):
        return

    # Contiguous stretches of included values (a complement may have two tails).
    breaks = np.flatnonzero(np.diff(selected) > 1)
    groups = np.split(selected, breaks + 1)
    half = (step / 2 if step is not None else 0)
    for group in groups:
        left = coordinates[group[0]] - half
        right = coordinates[group[-1]] + half
        if np.isclose(left, right):
            left -= .02 * (coordinates[-1] - coordinates[0])
            right += .02 * (coordinates[-1] - coordinates[0])
        # Arrow at the right end only: accumulate from smaller to larger values.
        ax.annotate("", xy=(right, y), xytext=(left, y),
                    arrowprops={"arrowstyle": "->", "color": color, "lw": 1.6})

    # When the included set is interrupted, do not visually include the gap.
    for previous, following in zip(groups[:-1], groups[1:]):
        left = coordinates[previous[-1]] + half
        right = coordinates[following[0]] - half
        ax.plot([left, right], [y, y], color=color, lw=1.3,
                ls=(0, (2, 4)), alpha=.65)

    ax.text((coordinates[selected[0]] + coordinates[selected[-1]]) / 2,
            y + ax.get_ylim()[1] * .025, label,
            ha="center", color=color, fontsize=9)


def plot_discrete_event(dist, values, included, title, xlabel, event_label, other_label):
    values = np.asarray(values)
    probs = dist.pmf(values)
    included = np.asarray(included, dtype=bool)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.bar(values[~included], probs[~included], color="lightgray", label=other_label)
    ax.bar(values[included], probs[included], color="steelblue", label=event_label)
    ax.set(xlabel=xlabel, ylabel="Probability", title=title)
    ax.set_xticks(values)
    ymax = max(probs) * 1.5
    ax.set_ylim(0, ymax)
    step = np.min(np.diff(values)) if len(values) > 1 else 1
    draw_probability_span(ax, values, included, ymax * .91, event_label, "steelblue", step=.7*step)
    draw_probability_span(ax, values, ~included, ymax * .77, other_label, "dimgray", step=.7*step)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_continuous_event(dist, x, included, title, xlabel, event_label, other_label):
    x = np.asarray(x)
    included = np.asarray(included, dtype=bool)
    density = dist.pdf(x)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(x, density, color="black", lw=1.5)
    ax.fill_between(x, density, where=included, color="steelblue", alpha=.55,
                    label=event_label)
    ax.fill_between(x, density, where=~included, color="lightgray", alpha=.9,
                    label=other_label)
    ymax = density.max() * 1.55
    ax.set(xlabel=xlabel, ylabel="Probability density", title=title)
    ax.set_ylim(0, ymax)
    draw_probability_span(ax, x, included, ymax * .9, event_label, "steelblue")
    draw_probability_span(ax, x, ~included, ymax * .76, other_label, "dimgray")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


## 1. Experiments, sample spaces, and events (~25 min)

A **random experiment** has an uncertain outcome. The **sample space** $S$ contains all possible outcomes. An **event** is a subset of $S$.

For a fair six-sided die, $S=\{1,2,3,4,5,6\}$.

- $A$: the result is even, $A=\{2,4,6\}$.
- $B$: the result is greater than 4, $B=\{5,6\}$.
- $A\cap B$: both events occur.
- $A\cup B$: at least one event occurs.

For **equally likely** outcomes (Laplace's rule), $P(A)=|A|/|S|$. This shortcut does **not** apply automatically to unequal probabilities.

In [ ]:
S = {1, 2, 3, 4, 5, 6}
A = {2, 4, 6}
B = {5, 6}

print("A ∩ B:", A & B)
print("A ∪ B:", A | B)
print(f"P(A): {len(A) / len(S):.3f}")
print(f"P(A or B): {len(A | B) / len(S):.3f}")

### Addition, intersection, and conditioning

For any events $A$ and $B$:

$$P(A\cup B)=P(A)+P(B)-P(A\cap B)$$

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)},\qquad P(B)>0$$

If $A$ and $B$ are **independent**, then $P(A\cap B)=P(A)P(B)$. Do not multiply probabilities simply because the question says **and**: independence must be justified.

**Example — playing cards:** In a standard 52-card deck, 26 cards are red and 4 are queens. Two queens are red. Therefore:

$$P(\text{red or queen})=\frac{26+4-2}{52}$$

$$P(\text{queen}\mid\text{red})=\frac{2}{26}$$

In [ ]:
p_red_or_queen = (26 + 4 - 2) / 52
p_queen_given_red = 2 / 26
print(f"P(red or queen) = {p_red_or_queen:.3f}")
print(f"P(queen | red) = {p_queen_given_red:.3f}")

### Check for Understanding 1 · 3 minutes

A fair die is rolled once. Find **by hand**:

1. $P(\text{even or greater than 4})$.
2. $P(\text{even}\mid\text{greater than 4})$.

Then use the sets `A` and `B` above to verify your answers.

In [ ]:
# Your turn
# p_union = ...
# p_conditional = ...

### Theoretical probability vs observed frequency

A fair coin has theoretical probability $P(H)=0.5$. In a finite experiment, the **observed proportion** of heads need not be exactly 0.5. It is an estimate, not the true probability.

With more independent flips of the same fair coin, the observed proportion tends to stabilize near 0.5 (Law of Large Numbers). This does **not** mean every short sequence must contain half heads.

### How the simulation works

We use **SciPy** to generate the random outcomes. `binom.rvs(n=1, p=0.5, size=200, random_state=42)` generates **200 independent Bernoulli trials**: each trial has one attempt (`n=1`) and a 50% probability of success (`p=0.5`). We encode heads as `1` and tails as `0`.

- `np.cumsum(flips)` counts the heads observed **so far**, after each flip.
- `np.arange(1, len(flips) + 1)` gives the number of flips completed so far: 1, 2, ..., 200.
- Dividing these arrays produces the **cumulative observed proportion** of heads after each flip, rather than dividing by 200 every time.

**Important distinction:** the *binomial distribution* is described by parameters $n$ and $p$; $k$ is the number of successes whose probability we want to evaluate. In `binom.rvs(n=1, p=0.5, size=200, random_state=42)`, `size` controls how many **independent simulated results** are generated. `random_state=42` makes the simulation reproducible. Later, `binom(n=10, p=0.4).pmf(k)` will evaluate the probability of **exactly $k$ successes** in 10 trials.


In [ ]:
# Generate 200 independent Bernoulli trials: 1 = heads, 0 = tails.
flips = binom.rvs(n=1, p=0.5, size=200, random_state=42)

# After each flip, count heads so far and divide by flips so far.
cumulative_heads = np.cumsum(flips)
number_of_flips = np.arange(1, len(flips) + 1)
proportion_heads = cumulative_heads / number_of_flips

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(number_of_flips, proportion_heads, label="Observed proportion")
ax.axhline(0.5, color="black", linestyle="--", label="Theoretical P(heads)")
ax.set(xlabel="Number of flips", ylabel="Proportion of heads", ylim=(0, 1))
ax.legend()
plt.show()

## 2. A short introduction to Bayes' theorem (~10 min)

Bayes' theorem helps us **update a probability when new evidence becomes available**:

$$P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}$$

### Example: a fraud-detection alert

Suppose an online store uses a system that flags potentially fraudulent transactions. Assume:

- **5%** of all transactions are fraudulent: $P(F)=0.05$.
- The system flags **90%** of fraudulent transactions: $P(+\mid F)=0.90$.
- It also flags **10%** of legitimate transactions by mistake: $P(+\mid \neg F)=0.10$.

**Notation:**

- $F$: the transaction is fraudulent.
- $\neg F$: the transaction is **not** fraudulent (it is legitimate).
- $+$: the system raises a positive fraud alert (not necessarily a real fraud).
- $P(A\mid B)$: probability of $A$ **given that** $B$ has occurred. The symbol $\mid$ means "given".
- $P(+\mid \neg F)$: probability of a **false positive**, an alert for a legitimate transaction.

**Question:** If a transaction is flagged, what is the probability that it is actually fraudulent?

First calculate the overall probability of an alert by considering **both ways an alert can occur**:

$$P(+)=P(+\mid F)P(F)+P(+\mid \neg F)P(\neg F)$$

Then apply Bayes' theorem to find $P(F\mid +)$. Notice that $P(+\mid F)$ and $P(F\mid +)$ are **not the same probability**.

*All rates in this example are hypothetical.*

In [ ]:
p_fraud = 0.05
p_alert_given_fraud = 0.90
p_alert_given_legitimate = 0.10

# An alert can come from a fraudulent OR a legitimate transaction.
p_alert = (
    p_alert_given_fraud * p_fraud
    + p_alert_given_legitimate * (1 - p_fraud)
)
p_fraud_given_alert = p_alert_given_fraud * p_fraud / p_alert

print(f"P(alert) = {p_alert:.3f}")
print(f"P(fraud | alert) = {p_fraud_given_alert:.3f}")

### Check for Understanding 2 · 3 minutes

Suppose the alert system becomes more selective: its **false-positive rate** drops from 10% to **5%**, while the fraud prevalence (5%) and detection rate (90%) stay unchanged.

1. Predict whether $P(F\mid +)$ will increase or decrease.
2. Recalculate the probability of fraud **given an alert**.
3. Why does improving the false-positive rate change the reliability of an alert?

In [ ]:
# Your turn
# p_alert_given_legitimate_new = ...
# p_alert_new = ...
# p_fraud_given_alert_new = ...

## 3. Random variables and probability functions (~25 min)

A **random variable** $X$ assigns a number to each outcome of an experiment.

- **Discrete:** possible values are countable, such as number of successful calls.
- **Continuous:** values may fall anywhere in an interval, such as waiting time.

| Function | Meaning | Typical question |
|---|---|---|
| `pmf(k)` | Probability at an exact **discrete** value | $P(X=k)$ |
| `pdf(x)` | Probability **density** at a continuous value | How concentrated is probability near $x$? |
| `cdf(x)` | Cumulative probability | $P(X\le x)$ |
| `sf(x)` | Survival probability, `1 - cdf(x)` | $P(X>x)$ |
| `ppf(q)` | Quantile (inverse CDF) | Which value has cumulative probability $q$? |

**Crucial:** For continuous variables, `pdf(x)` is **not** $P(X=x)$; that point probability is zero. Probabilities correspond to **areas under the density curve**.

**Discrete boundary:** For an integer-valued variable, $P(X\ge3)=P(X>2)=\operatorname{sf}(2)$, not `sf(3)`.

**Reading the plots:** In the examples below, one color marks outcomes **included** in the requested event, while another marks those **excluded**. The horizontal arrows label the corresponding probability sums (discrete) or areas (continuous). The two complementary regions always add up to **1**. For a continuous variable, an endpoint has probability zero, so $P(X<a)=P(X\le a)$; this is not generally true for a discrete variable.


In [ ]:
# One example of each question type
x = binom(n=10, p=0.4)
print(f"Exactly 3 successes: {x.pmf(3):.3f}")
print(f"At most 3 successes: {x.cdf(3):.3f}")
print(f"At least 3 successes: {x.sf(2):.3f}")
print(f"90th percentile: {x.ppf(0.90):.3f}")

### Check for Understanding 3 · 4 minutes

For the same binomial model, write **the appropriate SciPy expression** for:

1. $P(X=5)$
2. $P(X\le5)$
3. $P(X>5)$
4. $P(X\ge5)$

Which two expressions differ only because of the strict vs inclusive inequality?

In [ ]:
# Your turn
# exactly_five = ...
# at_most_five = ...
# more_than_five = ...
# at_least_five = ...

## 4. Discrete distributions (~35 min)

### 4.1 Binomial: successes in a fixed number of trials

**Question:** We contact 10 independent suppliers. Each has a 40% chance of meeting our requirements. What is the probability that **at least three** succeed?

Let $X\sim\mathrm{Binomial}(n=10,p=0.4)$. Assumptions: a fixed number of trials, two outcomes per trial, independence, and constant success probability.

To understand the probability of **exactly $k$ successes** in $n$ trials:

1. We need $k$ successes **and** $n-k$ failures. For any **particular ordering**, independence lets us multiply their probabilities: $p^k(1-p)^{n-k}$.
2. But we do **not** care *which* trials succeed. All orderings with exactly $k$ successes have the same probability.
3. We therefore multiply by the number of ways to choose which $k$ trials succeed: the **binomial coefficient** $\binom{n}{k}$.

Putting these ideas together:

$$P(X=k)=\binom{n}{k}p^k(1-p)^{n-k}$$

The binomial coefficient is:

$$\binom{n}{k}=\frac{n!}{k!(n-k)!}$$

The exclamation mark denotes a **factorial**:

$$n!=n\times(n-1)\times(n-2)\times\cdots\times2\times1,\qquad 0!=1$$

For example, $4!=4\times3\times2\times1=24$.

**At least three successes:** Because the probabilities of all possible outcomes add up to 1, split the outcomes into two complementary groups:

$$1=P(X\le2)+P(X\ge3)$$

Therefore,

$$\begin{aligned}P(X\ge3)&=P(X=3)+P(X=4)+\cdots+P(X=10)\\&=1-P(X\le2)\\&=1-\operatorname{CDF}(2)\\&=\operatorname{SF}(2).\end{aligned}$$

In general, for an **integer-valued** random variable:

$$P(X\ge k)=1-P(X\le k-1)=\operatorname{SF}(k-1).$$

Notice that the complement of $X\ge k$ is $X<k$, or equivalently $X\le k-1$ for integer values.

Here **$n$ and $p$ define the distribution**; **$k$ is the value at which we evaluate its PMF**. For example, `binom(n=10, p=0.4).pmf(3)` calculates $P(X=3)$, whereas `.sf(2)` calculates $P(X\ge3)$.


In [ ]:
suppliers = binom(n=10, p=0.40)
p_at_least_3 = suppliers.sf(2)
print(f"P(X >= 3) = 1 - CDF(2) = SF(2) = {p_at_least_3:.3f}")
print(f"P(X <= 2) = CDF(2) = {suppliers.cdf(2):.3f}")
print(f"Total = {p_at_least_3 + suppliers.cdf(2):.3f}")

k = np.arange(0, 11)
plot_discrete_event(suppliers, k, k >= 3,
    "Binomial: at least 3 successful suppliers", "Successful suppliers",
    "P(X >= 3) = 1 - CDF(2) = SF(2)", "P(X <= 2) = CDF(2)")

### Check for Understanding 4 · 3 minutes

If we contact **15** suppliers instead of 10, keeping $p=0.4$, how does $P(X\ge3)$ change? Predict first, then calculate.

In [ ]:
# Your turn
# probability_15 = ...

### 4.2 Geometric: trials until the first success

**Question:** Each independent job application has a 20% chance of yielding an interview invitation. What is the probability that the **first** invitation comes on application 3?

In SciPy, `geom` counts the **trial on which the first success occurs**, so values start at 1.

For the first success to occur on trial $k$, we must have **$k-1$ failures followed by one success**:

$$\underbrace{F, F, \ldots, F}_{k-1\text{ failures}},\ S$$

Assuming independent trials with constant success probability $p$, we multiply the probabilities:

$$P(X=k)=(1-p)^{k-1}p,\quad k=1,2,\ldots$$

Unlike the binomial case, **there is no combinatorial factor**: the order is fixed, because the first success must happen **on the $k$-th trial**. For $k=3$, the only possible sequence is **failure, failure, success**.


In [ ]:
applications = geom(p=0.20)
print(f"First invitation on application 3: {applications.pmf(3):.3f}")
print(f"First invitation after application 4: {applications.sf(4):.3f}")

# Exactly 3: one bar is included; the other bars are excluded.
k = np.arange(1, 21)
plot_discrete_event(applications, k, k == 3,
    "Geometric: first invitation on application 3", "Application number",
    "P(X = 3) = PMF(3)", "P(X != 3) = 1 - PMF(3)")
# Note: the chart shows 1..20 only; the geometric distribution has an infinite tail.

In [ ]:
plot_discrete_event(applications, k, k > 4,
    "Geometric: first invitation after application 4", "Application number",
    "P(X > 4) = 1 - CDF(4) = SF(4)", "P(X <= 4) = CDF(4)")

### Check for Understanding 5 · 3 minutes

What is the probability that the first invitation arrives **within the first four applications**? Write the event and use the correct SciPy method.

In [ ]:
# Your turn
# p_within_four = ...

### 4.3 Poisson: event counts in an interval

**Question:** An intersection experiences an average of two accidents per day. Under a Poisson model, what is the probability of **zero accidents tomorrow**? What about **five or more**?

$X\sim\mathrm{Poisson}(\mu=2)$. Here $\mu$ is the expected **count in the specified interval**. The model assumes an appropriate stable event rate and independent increments; it may be unsuitable if events cluster or the rate changes strongly.

In [ ]:
accidents = poisson(mu=2)
print(f"Zero accidents: {accidents.pmf(0):.3f}")
print(f"Five or more accidents: {accidents.sf(4):.3f}")

k = np.arange(0, 16)
plot_discrete_event(accidents, k, k >= 5,
    "Poisson: five or more accidents", "Accidents in one day",
    "P(X >= 5) = 1 - CDF(4) = SF(4)", "P(X <= 4) = CDF(4)")
# The chart truncates the tiny Poisson tail beyond 15; the reported SF includes it.
plot_discrete_event(accidents, k, k == 0,
    "Poisson: zero accidents", "Accidents in one day",
    "P(X = 0) = PMF(0)", "P(X != 0) = 1 - PMF(0)")

### Check for Understanding 6 · 3 minutes

Suppose the average is **three** accidents per day. What is $P(X\le2)$? Remember to update the model parameter.

In [ ]:
# Your turn
# p_two_or_fewer = ...

## 5. Continuous distributions (~35 min)

### 5.1 Exponential: waiting time between events

**Question:** Under a constant-rate Poisson arrival process, a bus arrives on average every 10 minutes. What is the probability that the next waiting time is **less than 8 minutes**? What is the probability it exceeds 15 minutes?

$T\sim\mathrm{Exponential}(\lambda)$, where $\lambda=0.1$ arrivals per minute. SciPy uses `scale = 1 / lambda`, which equals the **mean waiting time**.

$$f(t)=\lambda e^{-\lambda t},\quad t\ge0$$

The exponential model has a *memoryless* property. Real bus timetables often violate its assumptions; this is an illustrative model.

In [ ]:
mean_wait = 10  # minutes
waiting_time = expon(scale=mean_wait)
print(f"Wait at most 8 min: {waiting_time.cdf(8):.3f}")
print(f"Wait more than 15 min: {waiting_time.sf(15):.3f}")

minutes = np.linspace(0, 80, 801)
plot_continuous_event(waiting_time, minutes, minutes <= 8,
    "Exponential: wait at most 8 minutes", "Waiting time (minutes)",
    "P(T <= 8) = CDF(8)", "P(T > 8) = 1 - CDF(8) = SF(8)")
# For continuous distributions, P(T < 8) = P(T <= 8): the boundary has probability zero.
# Plot shown only to 80 min; theoretical probabilities include the infinite tail.
plot_continuous_event(waiting_time, minutes, minutes > 15,
    "Exponential: wait more than 15 minutes", "Waiting time (minutes)",
    "P(T > 15) = 1 - CDF(15) = SF(15)", "P(T <= 15) = CDF(15)")

### Check for Understanding 7 · 3 minutes

Calculate the probability that the waiting time is **between 5 and 15 minutes**. Hint: subtract two cumulative probabilities.

In [ ]:
# Your turn
# p_between_5_and_15 = ...

### 5.2 Normal: values around a mean

**Question:** Suppose student heights are approximately normally distributed with mean 175 cm and standard deviation 7.5 cm. What is the probability that a randomly selected student is taller than 182 cm? What is the probability of a height between 170 and 175 cm?

$H\sim\mathcal{N}(\mu=175,\sigma=7.5^2)$, where SciPy's `scale` parameter is the **standard deviation** $\sigma$, not the variance.

In [ ]:
heights = norm(loc=175, scale=7.5)
print(f"Taller than 182 cm: {heights.sf(182):.3f}")
print(f"Between 170 and 175 cm: {heights.cdf(175) - heights.cdf(170):.3f}")
print(f"90th percentile (cm): {heights.ppf(0.90):.3f}")

h = np.linspace(140, 210, 701)
plot_continuous_event(heights, h, h > 182,
    "Normal: height greater than 182 cm", "Height (cm)",
    "P(H > 182) = 1 - CDF(182) = SF(182)", "P(H <= 182) = CDF(182)")
# Plotting window is finite; SciPy probabilities include the full tails.
plot_continuous_event(heights, h, (h >= 170) & (h <= 175),
    "Normal: height between 170 and 175 cm", "Height (cm)",
    "P(170 <= H <= 175) = CDF(175) - CDF(170)",
    "Outside interval = 1 - inside")

### Check for Understanding 8 · 4 minutes

Find the height corresponding to the **top 5%** of this modeled population. Is the answer obtained using `cdf`, `sf`, or `ppf`? Explain your choice.

In [ ]:
# Your turn
# height_top_five_percent = ...

## 6. Choosing the model: quick recap (~10 min)

| Situation | Random variable | Model | Typical parameters |
|---|---|---|---|
| Number of successes in $n$ independent trials | Count of successes | Binomial | `n`, `p` |
| Trial of the first success | Trial number | Geometric | `p` |
| Events during a fixed interval | Count of events | Poisson | `mu` (mean count) |
| Time until the next event | Waiting time | Exponential | `scale` (mean waiting time) |
| Symmetric continuous measurements around a mean | Measurement | Normal | `loc` (mean), `scale` (SD) |

**Before computing anything, ask:** What does $X$ measure? What assumptions are we making? Does the question say *exactly*, *at most*, *at least*, *more than*, *between*, or *percentile*?

## Final recap challenge · 12–15 minutes

### A day at an online learning platform

The following scenarios are **hypothetical** and assume the stated models are reasonable.

**A. Outreach.** The platform contacts 12 potential partners. Each independently has a 25% chance of agreeing. Find the probability that **at least four** agree.

**B. Support.** Support receives an average of 3 tickets per hour under a Poisson model. Find the probability of **more than five** tickets in the next hour.

**C. Waiting time.** Assume support tickets arrive as a constant-rate Poisson process at 3 per hour. Find the probability that the time until the next ticket is **less than 10 minutes**. *(Check that your units match.)*

**D. Assessment scores.** Suppose assessment scores follow a normal distribution with mean 70 and standard deviation 10. Find the score that marks the **top 10%**.

For **each** question, write down:

1. What the random variable represents.
2. Which distribution and parameters you choose.
3. The event in mathematical notation.
4. The SciPy expression and numerical result.
5. One assumption that might be questionable in real data.

**Bonus (if time permits):** In B and C, explain why a Poisson **count** and an exponential **waiting time** can describe the same arrival process.

In [ ]:
# Final challenge — work here

# A. Outreach

# B. Support tickets

# C. Waiting time

# D. Assessment scores


## Key takeaways

- Probabilities describe a model of uncertainty; observed frequencies are estimates.
- `pmf` applies to exact discrete values; `pdf` describes continuous density, not point probability.
- `cdf` gives left-tail probability; `sf` gives right-tail probability; `ppf` finds a quantile.
- Match the **question and assumptions** to a distribution before choosing a function.
- A mathematically correct probability calculation can still be misleading if the model is unsuitable.

**Next:** Practice these ideas in the *Intro to Probability* lab. Hypothesis testing and sampling distributions come in the next lesson.